In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from dowhy import CausalModel
import statsmodels.api as sm # Import statsmodels to potentially inspect data if needed

# Load and preprocess 5K rows
data = pd.read_csv('/content/drive/MyDrive/DS Project/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv', nrows=500)
print('Data Loaded')

# Clean numerical columns
for col in ["Length of Stay", "Birth Weight", "Total Charges", "Total Costs"]:
    data[col] = pd.to_numeric(data[col].astype(str).str.replace(',', '', regex=False), errors='coerce')
    # Keep fillna for initial handling, but we'll drop NaNs later for safety
    data[col] = data[col].fillna(data[col].median())
print('Numerical Cleanup completed')

# Encode categorical variables
data = pd.get_dummies(data, drop_first=True)

# Define variables
outcome = "Total Costs"
treatment = "Length of Stay"
# Recalculate common_causes after get_dummies as new columns are added
common_causes = [col for col in data.columns if col != outcome and col != treatment]

# --- Added step to handle remaining NaN/inf values ---
# Check for and drop rows with NaN or inf values in relevant columns
# Only consider columns that will be used in the model (outcome, treatment, common_causes)
cols_to_check = [outcome, treatment] + common_causes
# Ensure that cols_to_check only contains columns actually present in the dataframe
cols_to_check = [col for col in cols_to_check if col in data.columns]

initial_rows = data.shape[0]
data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=cols_to_check)
rows_after_drop = data.shape[0]

if initial_rows != rows_after_drop:
    print(f"Dropped {initial_rows - rows_after_drop} rows containing NaN or inf values in outcome, treatment, or common causes columns.")

if data.empty:
    raise ValueError("DataFrame is empty after dropping rows with missing values.")

print('NaN/inf handling completed')
# --- End of added step ---


print('Starting Causal Model')

# Initialize causal model
model = CausalModel(
    data=data,
    treatment=treatment,
    outcome=outcome,
    common_causes=common_causes # Use the recalculated common_causes
)

# Identify and estimate effect
identified_estimand = model.identify_effect()
estimate = model.estimate_effect(identified_estimand, method_name="backdoor.linear_regression")
refutation = model.refute_estimate(identified_estimand, estimate, method_name="placebo_treatment_refuter")

# Save results to file
with open("causal_results.txt", "w") as f:
    f.write("=== CAUSAL ANALYSIS RESULTS ===\n\n")
    f.write(f"Causal Estimate (Length of Stay → Total Costs): {estimate.value}\n\n")
    f.write("Refutation Test:\n")
    f.write(str(refutation))

print("Causal analysis completed and results saved to causal_results.txt")

# Download file in Colab
#from google.colab import files
#files.download('causal_results.txt')

Data Loaded
Numerical Cleanup completed
Dropped 1 rows containing NaN or inf values in outcome, treatment, or common causes columns.
NaN/inf handling completed
Starting Causal Model


/usr/local/lib/python3.11/dist-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/usr/local/lib/python3.11/dist-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/usr/local/lib/python3.11/dist-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFra

Causal analysis completed and results saved to causal_results.txt


/usr/local/lib/python3.11/dist-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.4/398.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 58.4 MB/s eta 0:00:00
  Attempting uninstall: cython
    Found existing installation: Cython 3.0.12
    Uninstalling Cython-3.0.12:
      Successfully uninstalled Cython-3.0.12
